# HOW TO USE (read first)
1. Click **Run All** at the top. Leave everything as is. This is the practice run on fake data.
2. The first cell installs the libraries for you (no terminal needed). Wait for it.
3. At the bottom you should see **ALL CHECKS PASSED**.
4. Only after that: change `PRACTICE_MODE = False` and set `REAL_CODENET_PATH` in the Configuration cell.

In [ ]:
%pip install -q pandas numpy pyarrow beautifulsoup4 lxml

# WhoWroteIt? - Phase 1 - Person 2: CodeNet subsetting and extraction

**Outputs:** `codenet_subset.parquet`, `problems.parquet`, `codenet_log.csv`

**What this notebook does (in order):**
1. Load the clean AIGCodeSet (we need her IDs and code to remove overlap)
2. Pick problems: Pool A (the 317 AIGCodeSet problems) + Pool B (other problems)
3. Filter metadata: Python 3 only, 3 statuses, size range
4. Read the code files and apply the exact AIGCodeSet text cleaning
5. Filter by size, drop comment-only files, flag `parses_ok`
6. Remove duplicates inside CodeNet
7. Remove overlap with AIGCodeSet (3 layers)
8. Cap per (problem, status), assign `role`, sample to target size
9. Extract problem descriptions from HTML -> `problems.parquet`
10. Final checks + write files

In [ ]:
import os, re, ast, math, hashlib, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup

## 0. Configuration
**Only change `PRACTICE_MODE` and `REAL_CODENET_PATH`.** Everything else is already tuned
to match AIGCodeSet's data (her human median is ~302 chars, her AI max is 5,170 chars / 222 lines).

In [ ]:
# ---------------------------------------------------------------
# STEP A: leave PRACTICE_MODE = True for your first run (fake data, takes ~1 minute).
# STEP B: when practice works, set PRACTICE_MODE = False and fill in REAL_CODENET_PATH.
# ---------------------------------------------------------------
PRACTICE_MODE = True
REAL_CODENET_PATH = r"C:\path\to\Project_CodeNet"   # <-- only used when PRACTICE_MODE = False. Folder that contains metadata/, data/, problem_descriptions/

AIG_PATH = Path("aigcodeset_clean.parquet")           # output of the AIGCodeSet cleaning step (already in the repo folder)

if PRACTICE_MODE:
    import subprocess, sys
    # builds a fake mini CodeNet in ./mock (needs make_mock_codenet.py in the same folder)
    subprocess.run([sys.executable, "make_mock_codenet.py"], check=True)
    CODENET_ROOT = Path("mock/Project_CodeNet")
    OUT_DIR = Path("practice_output")                  # practice results go here, NOT mixed with real files
    TARGET_TRAIN_EXTRA = 5800
else:
    CODENET_ROOT = Path(REAL_CODENET_PATH)
    OUT_DIR = Path(".")
    TARGET_TRAIN_EXTRA = 20000                         # total 'train_extra' human rows (Pool A + Pool B)
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42                      # every random step uses this -> reproducible

STATUS_KEEP = ["Accepted", "Wrong Answer", "Runtime Error"]   # same spelling as AIGCodeSet
CAP_PER_STATUS = 5             # per (problem, status) -> 15 per problem, same as AIGCodeSet
RAG_EXTRA_PER_PROBLEM = 5      # extra Accepted-only rows per problem for the RAG corpus (0 = disable)
OVERSAMPLE = 3                 # read 3x more candidates than needed, because filters will drop some

MIN_CHARS, MAX_CHARS = 50, 5000    # AIGCodeSet's AI max is 5,170
MIN_LINES, MAX_LINES = 3, 200      # AIGCodeSet's AI max is 222
PY3_REGEX = r"^(Python \(3|PyPy3)"  # Python 3 / PyPy3 only (Python 2 breaks ast.parse)
POOL_B_DATASET = "AtCoder"          # AIGCodeSet problems are AtCoder; set None to allow any. Check: problem_list.dataset.value_counts()

rng = np.random.default_rng(SEED)
log = {}   # row counts after every step -> codenet_log.csv (goes in the report)

## 1. Shared helper functions
`clean_text` is **exactly the AIGCodeSet cleaning**. If we hashed code without it, identical code would
get different hashes and the overlap check would silently miss it.

In [ ]:
def clean_text(s: str) -> str:
    """the AIGCodeSet cleaning: line endings, BOM, NUL."""
    return (str(s).replace("\r\n", "\n").replace("\r", "\n")
                  .replace("\ufeff", "").replace("\x00", ""))

def loose_text(s: str) -> str:
    """Looser version: strip trailing spaces on each line + strip the whole file.
    Catches copies that differ only in whitespace."""
    return "\n".join(line.rstrip() for line in s.split("\n")).strip()

def md5(s: str) -> str:
    return hashlib.md5(s.encode("utf-8")).hexdigest()

def has_real_code(s: str) -> bool:
    """False if the file is only blank lines / comments."""
    return any(l.strip() and not l.strip().startswith("#") for l in s.split("\n"))

def parses_ok(s: str) -> bool:
    """True if the code is valid Python 3 syntax (Yassin needs this for AST features)."""
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            ast.parse(s)
        return True
    except (SyntaxError, ValueError, RecursionError, MemoryError):
        return False

## 2. Load the clean AIGCodeSet
We need three things from it, to remove overlap later:
- her **problem IDs** (= Pool A)
- her **submission IDs** - *both* human rows *and* AI rows (AI rows store the ID of the buggy
  submission the LLM fixed; the from-scratch ones say `"unrelated"`, which we ignore)
- her **code hashes** (raw and loose)

In [ ]:
aig = pd.read_parquet(AIG_PATH)
print(aig.shape, "| problems:", aig.problem_id.nunique())

aig_problems = set(aig.problem_id)
aig_sub_ids  = set(aig.submission_id) - {"unrelated"}
aig_hash_raw   = {md5(clean_text(c)) for c in aig.code_raw}
aig_hash_loose = {md5(loose_text(clean_text(c))) for c in aig.code_raw}

log["aig_rows"] = len(aig)
log["aig_problems"] = len(aig_problems)
log["aig_submission_ids"] = len(aig_sub_ids)

## 3. Pick problems: Pool A and Pool B
- **Pool A** = the 317 AIGCodeSet problems. These have BOTH human and AI samples, so val/test must come only from here.
- **Pool B** = other problems -> extra variety for training and RAG only (no AI samples exist for them).

To avoid reading metadata for all ~4,000 problems, we only load a random slice of Pool B,
just large enough to reach the target.

In [ ]:
problem_list = pd.read_csv(CODENET_ROOT / "metadata" / "problem_list.csv").rename(columns={"id": "problem_id"})
print(problem_list.columns.tolist())
if "dataset" in problem_list:
    print(problem_list["dataset"].value_counts().to_dict())

pool_a = sorted(aig_problems)
cand_b = problem_list if POOL_B_DATASET is None or "dataset" not in problem_list \
         else problem_list[problem_list["dataset"] == POOL_B_DATASET]
pool_b_all = sorted(set(cand_b.problem_id) - aig_problems)
# Pool B problems MUST have an HTML description (RAG needs the explanation text)
pool_b_all = [p for p in pool_b_all if (CODENET_ROOT / "problem_descriptions" / f"{p}.html").exists()]

budget_b = max(TARGET_TRAIN_EXTRA - len(pool_a) * 3 * CAP_PER_STATUS, 0)       # rows we still need from Pool B
n_b_needed = math.ceil(budget_b / (3 * CAP_PER_STATUS) * 1.6) + 10            # 1.6x safety margin (some problems fail filters)
pool_b = list(rng.permutation(pool_b_all)[:n_b_needed])

print(f"Pool A: {len(pool_a)} problems | Pool B candidates: {len(pool_b)} of {len(pool_b_all)}")
log["pool_a_problems"] = len(pool_a)
log["pool_b_candidate_problems"] = len(pool_b)

## 4. Metadata filtering (cheap - no code files read yet)
Keep: language Python, **Python 3 only**, the 3 statuses, rough size range (`code_size` is in bytes).
Then drop every submission ID that exists in AIGCodeSet (**overlap layer 1**).

In [ ]:
META_COLS = ["submission_id", "problem_id", "language", "original_language",
             "filename_ext", "status", "code_size"]
counters = dict(all_rows=0, python=0, python3=0, status_ok=0, size_ok=0)

def load_candidates(pids):
    frames = []
    for i, pid in enumerate(pids):
        p = CODENET_ROOT / "metadata" / f"{pid}.csv"
        if not p.exists():
            continue
        m = pd.read_csv(p, usecols=META_COLS)
        counters["all_rows"] += len(m)
        m = m[m["language"] == "Python"];                                  counters["python"]  += len(m)
        m = m[m["original_language"].str.match(PY3_REGEX, na=False)];     counters["python3"] += len(m)
        m = m[m["status"].isin(STATUS_KEEP)];                             counters["status_ok"] += len(m)
        # bytes >= chars, so MIN is safe; MAX gets 2x slack for non-ASCII. Exact char filter comes later.
        m = m[m["code_size"].between(MIN_CHARS, MAX_CHARS * 2)];          counters["size_ok"] += len(m)
        frames.append(m)
        if (i + 1) % 500 == 0:
            print(f"  metadata: {i+1}/{len(pids)} problems")
    return pd.concat(frames, ignore_index=True)

meta = load_candidates(pool_a + pool_b)
meta["in_aigcodeset_problem"] = meta["problem_id"].isin(aig_problems)
for k, v in counters.items():
    log[f"meta_{k}"] = v

n0 = len(meta)
meta = meta[~meta["submission_id"].isin(aig_sub_ids)]                    # OVERLAP LAYER 1
log["overlap_L1_submission_id_removed"] = n0 - len(meta)
print(counters, "| removed by submission_id:", n0 - len(meta))

## 5. Choose which files to read (oversampled, per problem+status)
Shuffle, then keep the first `needed * OVERSAMPLE` of each (problem, status) group.
Accepted needs extra rows for the RAG-only role.

In [ ]:
meta = meta.sample(frac=1, random_state=SEED).reset_index(drop=True)
meta["_rank"] = meta.groupby(["problem_id", "status"]).cumcount()
need = meta["status"].map({"Accepted": CAP_PER_STATUS + RAG_EXTRA_PER_PROBLEM,
                           "Wrong Answer": CAP_PER_STATUS, "Runtime Error": CAP_PER_STATUS})
cands = meta[meta["_rank"] < need * OVERSAMPLE].drop(columns="_rank").reset_index(drop=True)
log["candidates_to_read"] = len(cands)
print("files to read:", len(cands))

## 6. Read the code files + apply the AIGCodeSet cleaning

In [ ]:
def read_code(row):
    p = CODENET_ROOT / "data" / row.problem_id / "Python" / f"{row.submission_id}.{row.filename_ext}"
    try:
        return p.read_text(encoding="utf-8", errors="replace")
    except OSError:
        return None

cands["code_raw"] = [read_code(r) for r in cands.itertuples()]
n0 = len(cands)
cands = cands.dropna(subset=["code_raw"])
log["unreadable_files_removed"] = n0 - len(cands)

cands["code_raw"] = cands["code_raw"].map(clean_text)
n0 = len(cands)
cands = cands[~cands["code_raw"].str.contains("\ufffd", regex=False)]     # broken encoding
log["bad_encoding_removed"] = n0 - len(cands)
log["after_read_and_clean"] = len(cands)

## 7. Size filters, comment-only files, `parses_ok`
`n_chars` / `n_lines` use **the same formulas as AIGCodeSet**.

In [ ]:
cands["n_chars"] = cands["code_raw"].str.len()
cands["n_lines"] = cands["code_raw"].str.count("\n") + 1

n0 = len(cands)
cands = cands[cands["n_chars"].between(MIN_CHARS, MAX_CHARS) & cands["n_lines"].between(MIN_LINES, MAX_LINES)]
log["size_filter_removed"] = n0 - len(cands)

n0 = len(cands)
cands = cands[cands["code_raw"].map(has_real_code)]
log["comment_or_blank_only_removed"] = n0 - len(cands)

cands["parses_ok"] = cands["code_raw"].map(parses_ok)
# syntax errors are only plausible in Runtime Error submissions; in Accepted/WA they signal junk
n0 = len(cands)
cands = cands[cands["parses_ok"] | (cands["status"] == "Runtime Error")]
log["unparsable_non_RE_removed"] = n0 - len(cands)
log["after_size_and_syntax"] = len(cands)
print(log["after_size_and_syntax"], "rows | parses_ok rate:", round(cands.parses_ok.mean(), 3))

## 8. Dedup inside CodeNet, then overlap layers 2 and 3 against AIGCodeSet
- hash 1 = raw (after the AIGCodeSet cleaning) - exact match
- hash 2 = loose (trailing spaces stripped) - near-exact match

In [ ]:
cands["code_hash_raw"]   = cands["code_raw"].map(md5)
cands["_hash_loose"]     = cands["code_raw"].map(lambda s: md5(loose_text(s)))

n0 = len(cands)
cands = cands.drop_duplicates("code_hash_raw").drop_duplicates("_hash_loose")
log["internal_duplicates_removed"] = n0 - len(cands)

n0 = len(cands)
cands = cands[~cands["code_hash_raw"].isin(aig_hash_raw)]                # OVERLAP LAYER 2
log["overlap_L2_raw_hash_removed"] = n0 - len(cands)

n0 = len(cands)
cands = cands[~cands["_hash_loose"].isin(aig_hash_loose)]                # OVERLAP LAYER 3
log["overlap_L3_loose_hash_removed"] = n0 - len(cands)
log["after_overlap_removal"] = len(cands)

## 9. Cap per (problem, status), assign `role`, sample to target size
- first `CAP_PER_STATUS` rows of each group -> `train_extra` (status-balanced, like AIGCodeSet)
- next `RAG_EXTRA_PER_PROBLEM` **Accepted** rows -> `rag_only` (Yassin must NOT train on these)
- Pool A is always kept. Pool B problems are added (in random order) until `TARGET_TRAIN_EXTRA` is reached.
- Pool B problems must have all 3 statuses, so the status mix stays balanced.

In [ ]:
cands = cands.sample(frac=1, random_state=SEED).reset_index(drop=True)
cands["_rank"] = cands.groupby(["problem_id", "status"]).cumcount()
cands["role"] = np.where(cands["_rank"] < CAP_PER_STATUS, "train_extra",
                 np.where((cands["status"] == "Accepted") &
                          (cands["_rank"] < CAP_PER_STATUS + RAG_EXTRA_PER_PROBLEM), "rag_only", "drop"))
cands = cands[cands["role"] != "drop"]

a = cands[cands["in_aigcodeset_problem"]]
b = cands[~cands["in_aigcodeset_problem"]]

# Pool B: keep only problems that have all 3 statuses among train_extra rows
b_tr = b[b["role"] == "train_extra"]
ok_b = b_tr.groupby("problem_id")["status"].nunique()
ok_b = ok_b[ok_b == 3].index
b = b[b["problem_id"].isin(ok_b)]

# add Pool B problems in random order until the budget is used
budget = TARGET_TRAIN_EXTRA - int((a["role"] == "train_extra").sum())
sizes = b[b["role"] == "train_extra"].groupby("problem_id").size()
order = rng.permutation(sizes.index.to_numpy())
cum = np.cumsum(sizes.loc[order].to_numpy())
keep_b = set(order[cum <= max(budget, 0)])
b = b[b["problem_id"].isin(keep_b)]

final = pd.concat([a, b], ignore_index=True)
log["final_pool_a_problems"] = final[final.in_aigcodeset_problem].problem_id.nunique()
log["final_pool_b_problems"] = final[~final.in_aigcodeset_problem].problem_id.nunique()
print(final.groupby(["in_aigcodeset_problem", "role", "status"]).size())

## 10. Build the final schema (AIGCodeSet's column names + our additions)

In [ ]:
final["sample_id"]   = "cn_" + final["submission_id"]
final["source"]      = "codenet"
final["language"]    = "python"
final["label"]       = 0
final["llm_name"]    = "human"
final["scenario"]    = "human"
final["rag_eligible"] = final["status"] == "Accepted"

cols = ["sample_id", "source", "submission_id", "problem_id", "language", "code_raw",
        "label", "llm_name", "scenario", "status", "n_lines", "n_chars",
        "parses_ok", "in_aigcodeset_problem", "rag_eligible", "role", "code_hash_raw"]
codenet_subset = final[cols].sort_values(["problem_id", "status", "sample_id"]).reset_index(drop=True)

## 11. Problem descriptions: HTML -> clean text (`problems.parquet`)
Needed for every problem in our subset **and** every AIGCodeSet problem (RAG explanations).
Tricky parts: `<sup>`/`<sub>` (so 10^5 does not become 105) and inline tags that would otherwise break lines.

In [ ]:
INLINE_TAGS = ["var", "span", "strong", "b", "em", "i", "code", "a", "u", "font"]

def html_to_text(html: str) -> str:
    soup = BeautifulSoup(html, "lxml")
    for t in soup(["script", "style"]):
        t.decompose()
    for t in soup.find_all("sup"):
        t.replace_with("^" + t.get_text())
    for t in soup.find_all("sub"):
        t.replace_with("_" + t.get_text())
    for t in soup.find_all(INLINE_TAGS):
        t.unwrap()                      # keep the text, remove the tag (no line break)
    soup.smooth()                       # merge neighbouring text pieces
    text = soup.get_text("\n")
    lines = [re.sub(r"[ \t]+", " ", l).strip() for l in text.split("\n")]
    text = "\n".join(lines)
    return re.sub(r"\n{3,}", "\n\n", text).strip()

def get_title(html: str, fallback: str) -> str:
    h1 = BeautifulSoup(html, "lxml").find("h1")
    return h1.get_text(strip=True) if h1 else fallback

need_problems = sorted(set(codenet_subset.problem_id) | aig_problems)
pl = problem_list.set_index("problem_id")
rows, missing = [], []
for pid in need_problems:
    p = CODENET_ROOT / "problem_descriptions" / f"{pid}.html"
    if not p.exists():
        missing.append(pid); continue
    html = p.read_text(encoding="utf-8", errors="replace")
    info = pl.loc[pid] if pid in pl.index else {}
    g = lambda k: info.get(k) if hasattr(info, "get") else None
    desc = html_to_text(html)
    rows.append({
        "problem_id": pid,
        "title": get_title(html, str(g("name") or pid)),
        "description": desc,
        "n_chars": len(desc),
        "source_dataset": g("dataset"),
        "time_limit": g("time_limit"),
        "memory_limit": g("memory_limit"),
        "in_aigcodeset_problem": pid in aig_problems,
    })
problems = pd.DataFrame(rows)
log["problems_needed"], log["problems_with_description"], log["problems_missing_description"] = \
    len(need_problems), len(problems), len(missing)
print("descriptions:", len(problems), "| missing:", len(missing), missing[:5])
print(problems.iloc[0]["description"][:400])

## 12. Final checks (these `assert`s must all pass before you push)

In [ ]:
cs = codenet_subset
assert cs["sample_id"].is_unique
assert (cs["label"] == 0).all() and (cs["source"] == "codenet").all()
assert set(cs["status"]) <= set(STATUS_KEEP)
assert cs["code_raw"].notna().all() and (cs["code_raw"].str.strip() != "").all()
assert cs["n_chars"].between(MIN_CHARS, MAX_CHARS).all()

# zero overlap with AIGCodeSet, all 3 layers
assert not cs["submission_id"].isin(aig_sub_ids).any(),                      "overlap L1 (submission_id)"
assert not cs["code_hash_raw"].isin(aig_hash_raw).any(),                     "overlap L2 (raw hash)"
assert not cs["code_raw"].map(lambda s: md5(loose_text(s))).isin(aig_hash_loose).any(), "overlap L3 (loose hash)"

# no Python 2 (original_language filter) -> only RE rows may fail to parse
assert cs.loc[~cs["parses_ok"], "status"].eq("Runtime Error").all()

# every problem in the subset has a description
assert set(cs["problem_id"]) <= set(problems["problem_id"]), "subset problem without description"
if missing:
    print("WARNING: AIGCodeSet problems without description:", missing)

# rag_only rows must be Accepted
assert (cs.loc[cs.role == "rag_only", "status"] == "Accepted").all()

print("ALL CHECKS PASSED")
print("\nlength comparison (AIGCodeSet human vs CodeNet extra):")
print(pd.DataFrame({"aig_human": aig[aig.label == 0].n_chars.describe(),
                    "codenet_extra": cs.n_chars.describe()}).round(0))
print("\nstatus mix (train_extra):\n", cs[cs.role == "train_extra"].status.value_counts(normalize=True).round(3))

## 13. Save

In [ ]:
log["final_rows"] = len(cs)
for (role, status), n in cs.groupby(["role", "status"]).size().items():
    log[f"final_{role}_{status.replace(' ', '_')}"] = int(n)
log["final_pool_a_rows"] = int(cs.in_aigcodeset_problem.sum())
log["final_pool_b_rows"] = int((~cs.in_aigcodeset_problem).sum())

cs.to_parquet(OUT_DIR / "codenet_subset.parquet", index=False)
problems.to_parquet(OUT_DIR / "problems.parquet", index=False)
pd.Series(log).to_csv(OUT_DIR / "codenet_log.csv", header=["count"])
print(pd.Series(log).to_string())